In [ ]:
import sys
import os

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_rds, write_rds

In [41]:
df = read_rds("/Volumes/BCross/av_datasets_experiments/weighted_ngram_tracing_concat/combined_scores_raw.rds")

In [42]:
def get_completed_problems(df):
    """Filter out the dataframe to just include completed problems"""
    
    completed_df = (
        df[df['completed'] == True]
        .drop(columns=['num_problems', 'ngrams_found', 'completed'])
        .copy()
    )
        
    return completed_df

In [43]:
def get_distinct_problems(
    df,
    group_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens', 'min_token_size']
):
    
    distinct_levels = (
        df
        .groupby(group_cols, dropna=False)
        .size()
        .reset_index(name='problem_count')
    )
    
    return distinct_levels

In [44]:
def like_for_like(df, base_cols, compare_cols, id_col="problem"):
    """
    Within each base_cols group, keep only rows whose id_col appears
    in all distinct compare_cols combinations.
    """
    work = df.copy()

    all_group_cols = base_cols + compare_cols

    # Replace NaN in grouping columns so matching is consistent
    for col in all_group_cols:
        work[col] = work[col].astype("object").where(work[col].notna(), "__MISSING__")

    # Number of compare combinations required within each base group
    required = (
        work[base_cols + compare_cols]
        .drop_duplicates()
        .groupby(base_cols, dropna=False)
        .size()
        .reset_index(name="required_n")
    )

    # Number of compare combinations observed for each problem
    observed = (
        work[base_cols + compare_cols + [id_col]]
        .drop_duplicates()
        .groupby(base_cols + [id_col], dropna=False)
        .size()
        .reset_index(name="observed_n")
    )

    # Problems present in all compare combinations
    keep = (
        observed
        .merge(required, on=base_cols, how="left")
        .loc[lambda x: x["observed_n"] == x["required_n"], base_cols + [id_col]]
        .drop_duplicates()
    )

    # Filter original df using the same filled version
    out = work.merge(keep, on=base_cols + [id_col], how="inner")

    # Put missing values back
    for col in all_group_cols:
        out[col] = out[col].replace("__MISSING__", pd.NA)

    return out

In [69]:
def add_adjusted_token_size(
    df,
    distinct_problems,
    base_cols=['data_type', 'corpus', 'scoring_model', 'max_context_tokens']
):
    """Calculates the adjusted token size e.g. if a problem has min_token_size 4
    but we have problems with min_token_size 3 then the problem with 4 will now also be included.
    """
    # thresholds (the "new grouping") from distinct_levels
    levels = distinct_problems[base_cols + ['min_token_size']].rename(
        columns={'min_token_size': 'min_token_size_adjusted'}
    )
    
    tmp = df.merge(levels, on=base_cols, how='inner')

    # filter: df min_token_size >= threshold (adjusted bucket)
    tmp = tmp[tmp['min_token_size'] >= tmp['min_token_size_adjusted']]

    # partition by group + adjusted + problem, order by original min_token_size, take first
    tmp = tmp.sort_values(
        base_cols + ['min_token_size_adjusted', 'problem', 'min_token_size'],
        kind='mergesort'
    )

    tmp['row_number'] = (
        tmp.groupby(base_cols + ['min_token_size_adjusted', 'problem'], dropna=False)
        .cumcount()
        .add(1)
    )

    out = tmp[tmp['row_number'] == 1].drop(columns=['row_number']).copy()

    out = out.drop(columns=['min_token_size'])
    out = out.rename(columns={'min_token_size_adjusted': 'min_token_size'})
    
    return out

In [70]:
def adjusted_token_size_pipeline(
    df,
    group_cols=["data_type", "corpus", "scoring_model", "min_token_size", "weight", "alpha", "base"],
    base_cols=["data_type", "corpus", "scoring_model", "weight", "alpha", "base"]
):
    
    completed_df = get_completed_problems(df)
    
    distinct_levels = get_distinct_problems(completed_df, group_cols=group_cols)
    
    adjusted_token_size_df = add_adjusted_token_size(completed_df, distinct_levels, base_cols)
    
    return adjusted_token_size_df

In [71]:
adjusted_df = adjusted_token_size_pipeline(df)

In [75]:
like_df = like_for_like(
    adjusted_df,
    base_cols=["data_type", "corpus", "scoring_model", "min_token_size"],
    compare_cols=["weight", "alpha", "base"],
    id_col="problem"
)